# Phase 1 Closure — 01b: Four-Arm Production-Shape VRAM Go/No-Go Gate

**Phase:** Phase 1 closure (ADR-014 / ADR-018 / ADR-012). **Phase 2 has not started.**

This notebook implements the four-arm VRAM go/no-go gate specified in
`docs/proposals/phase1_closure_prereg.md` §8.2, to be executed on Colab's
**free-tier T4** (ADR-018). It measures whether QLoRA fine-tuning of
`Qwen/Qwen3-VL-4B-Instruct` fits in a T4's ~16 GiB across four production-shape
scenarios (load, train, resume, eval), and produces a GO/NO-GO verdict plus the
ADR-012 approval-gate evidence (the real trainable-parameter count and module
names, obtained by actually building the model).

## Non-goals — read before running

- **This notebook never loads CORD v2 data, images, or ground truth.** Every
  tensor used below (`input_ids`, `pixel_values`, `image_grid_thw`, `labels`)
  is synthetic / random, sized only to match the real processor's production
  output *shapes* (per `configs/derived_budget.yaml`, written by the companion
  notebook `notebooks/01a_closure_measurements.ipynb`). The gate measures
  **memory**, not correctness.
- This notebook does **not** start Phase 2, and does not create
  `notebooks/02_baseline.ipynb` or any Phase-2 code.
- This notebook does not modify `src/vlm_lab/*.py`, `tests/*`,
  `pyproject.toml`, `docs/*`, or `notebooks/01a_closure_measurements.ipynb`.

## Architecture note: `scripts/_vram_gate_common.py`

The model-construction logic (§7.3's frozen load → `prepare_model_for_kbit_training`
→ `get_peft_model` → `use_cache=False` order, plus the ADR-012 assertions), the
§8.2 measurement protocol (`measured_arm`), and the synthetic-batch builders
live in `scripts/_vram_gate_common.py`, imported below as `vgc`. This keeps
this notebook a thin orchestration layer (`AGENTS.md` §21) rather than
hundreds of lines of reusable logic in cells. It lives under `scripts/` rather
than `src/vlm_lab/` for two reasons: this task's scope explicitly excludes
modifying `src/vlm_lab/*.py`, and arm C genuinely needs this logic importable
from two standalone subprocess scripts
(`scripts/_vram_gate_c1_save.py`, `scripts/_vram_gate_c2_resume.py`), not only
from a notebook kernel.

## Arm isolation: a stated deviation from the frozen protocol

§8.2 step 1 specifies that **each arm runs in a fresh OS process**. Arms A, B,
and D below instead run inside this one shared notebook kernel, with explicit
`del` + `gc.collect()` + `torch.cuda.empty_cache()` +
`torch.cuda.reset_peak_memory_stats()` between them
(`vgc.free_cuda_memory(...)`). **This is an approximation, stated here
plainly rather than silently claimed as met** — CUDA context overhead,
allocator fragmentation, and cached allocations from an earlier arm are not
guaranteed to be fully released between arms within one process. True
process isolation would require separate notebook runs or subprocess
launches for every arm.

**Arm C is the one exception**, and is implemented as two genuinely separate
subprocesses (`scripts/_vram_gate_c1_save.py`, `scripts/_vram_gate_c2_resume.py`)
launched via `subprocess.run([sys.executable, ...])`, per §8.2's explicit
carve-out: "a resume that never left the process proves nothing about
reload."


## Environment setup

This notebook is meant to be runnable on its own, from a fresh Colab runtime,
via **Restart & Run All** — it does not assume any other notebook already ran
in the same session. The next two cells detect Colab and, only when running
there, obtain the repository and install this project's package (`vlm_lab`),
following the exact pattern used by `notebooks/01_dataset.ipynb` and
`notebooks/00_environment.ipynb`.


In [ ]:
# Detect whether this notebook is running on Google Colab.
# This gates the Colab-only setup cell below, mirroring notebooks/01_dataset.ipynb.
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Google Colab: {IN_COLAB}")


In [ ]:
REPO_URL = "https://github.com/Mr-Kondo/finetuning_vlm.git"

# The split-scoped loader work (load_development_splits / mechanical_access /
# sealed_test) lives on an unmerged branch, so a plain clone of the default
# branch would check out a `src/vlm_lab/` that predates it. That mismatch is
# invisible until an `ImportError` fires several cells later, because the
# notebook and the code it imports come from different places: the notebook
# from wherever you opened it, the code from whatever this clone checks out.
# TODO: reset to "" once this branch is merged into main.
GIT_REF = "worktree-phase2-gate"  # "" = default branch

if IN_COLAB:
    import importlib
    import importlib.util
    import os
    import re
    import site
    import subprocess
    import sys
    import tomllib

    if not REPO_URL:
        raise RuntimeError(
            "IN_COLAB is True but REPO_URL is not set. Set REPO_URL above before "
            "running this notebook on Colab -- the very next cell depends on "
            "`vlm_lab` being installed."
        )

    repo_dir = "repo"

    if os.path.isdir(repo_dir):
        print(f"'{repo_dir}' already exists -- updating instead of re-cloning "
              "(idempotent across kernel restarts).")
        fetch = subprocess.run(
            ["git", "-C", repo_dir, "fetch", "origin"], capture_output=True, text=True
        )
        if fetch.returncode != 0:
            raise RuntimeError(
                f"git fetch failed (exit code {fetch.returncode}):\n{fetch.stderr}"
            )
        if GIT_REF:
            checkout = subprocess.run(
                ["git", "-C", repo_dir, "checkout", GIT_REF], capture_output=True, text=True
            )
            if checkout.returncode != 0:
                raise RuntimeError(
                    f"git checkout {GIT_REF!r} failed (exit code {checkout.returncode}):\n"
                    f"{checkout.stderr}"
                )
        pull = subprocess.run(
            ["git", "-C", repo_dir, "pull"], capture_output=True, text=True
        )
        if pull.returncode != 0:
            raise RuntimeError(
                f"git pull failed (exit code {pull.returncode}):\n{pull.stderr}"
            )
    else:
        clone_cmd = ["git", "clone"]
        if GIT_REF:
            clone_cmd += ["--branch", GIT_REF]
        clone_cmd += [REPO_URL, repo_dir]
        clone = subprocess.run(clone_cmd, capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit code {clone.returncode}):\n{clone.stderr}"
            )

    os.chdir(repo_dir)

    # Parse pyproject.toml's exact `==` pins directly, rather than duplicating
    # version strings in this notebook, so the two can never drift apart.
    with open("pyproject.toml", "rb") as f:
        _pyproject = tomllib.load(f)
    _pinned_versions: dict[str, str] = {}
    for _dep in _pyproject["project"]["dependencies"]:
        _match = re.match(r"^([A-Za-z0-9_.-]+)==([A-Za-z0-9_.+-]+)$", _dep.strip())
        if _match:
            _pinned_versions[_match.group(1).lower()] = _match.group(2)

    # Use `sys.executable -m pip`, not a bare `pip` command: Colab can have
    # more than one Python/pip on PATH, and a bare `pip` invocation can
    # install into a different interpreter's site-packages than the one
    # actually running this notebook.
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
        capture_output=True, text=True,
    )
    print("--- pip install output ---")
    print(install.stdout)
    if install.stderr:
        print(install.stderr)
    print("--- end pip install output ---")
    if install.returncode != 0:
        raise RuntimeError(f"pip install failed (exit code {install.returncode}).")

    # This notebook also needs peft / bitsandbytes / accelerate directly (not
    # only via vlm_lab); pyproject.toml's dependencies already cover them, so
    # this install call is sufficient -- verified explicitly a few cells down.

    # torchaudio is not a dependency of this project; Colab's base VM image
    # pre-installs it compiled against whichever CUDA version that image
    # shipped with, which can hard-fail an unrelated import when the pinned
    # `torch` build resolves to a different CUDA version. Removed for
    # consistency with notebooks/00_environment.ipynb / notebooks/01_dataset.ipynb.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
        capture_output=True, text=True,
    )

    # Colab's own kernel startup imports PIL before any of this notebook's
    # cells run, and torch/torchvision may already be loaded too. Files just
    # reinstalled above on disk do NOT retroactively fix an already-imported
    # module in this same process -- especially for packages with compiled C
    # extensions, which cannot be safely hot-reloaded. Compare the
    # already-imported versions against the pins parsed above; if they don't
    # match, trigger a real interpreter restart and let the next kernel start
    # pick up the now-correct files.
    _version_check_targets = {"torch": "torch", "torchvision": "torchvision", "PIL": "pillow"}
    _mismatched = []
    for _module_name, _pin_key in _version_check_targets.items():
        if _module_name not in sys.modules:
            continue
        _installed = getattr(sys.modules[_module_name], "__version__", None)
        _pinned = _pinned_versions.get(_pin_key)
        if _installed is None or _pinned is None:
            continue
        if _installed.split("+")[0] != _pinned:
            _mismatched.append((_module_name, _installed, _pinned))

    if _mismatched:
        print(f"Already-imported package(s) do not match the pinned versions: {_mismatched}")
        print(
            "The correct versions are now installed on disk, but this running "
            "kernel already has the old ones cached in memory and cannot use "
            "the new files without an interpreter restart. Restarting the "
            "kernel now -- after it reconnects, run this notebook again "
            "(e.g. Restart & Run All); the git clone/pip install above will "
            "be fast no-ops since the files are already correct."
        )
        os.kill(os.getpid(), 9)

    # An editable install performed *after* this interpreter already started
    # writes a new `.pth` file into site-packages, but Python's `site` module
    # only processes `.pth` files at interpreter startup -- so `import
    # vlm_lab` (and `import peft` / `import bitsandbytes`, if this is their
    # first import too) can still fail right after install with a plain
    # ModuleNotFoundError. `site.addsitedir()` re-processes each
    # site-packages directory, including the newly written `.pth` file.
    for site_dir in site.getsitepackages():
        site.addsitedir(site_dir)
    importlib.invalidate_caches()

    try:
        importlib.import_module("vlm_lab")
    except ImportError as exc:
        raise RuntimeError(
            "Repository was cloned and `pip install -e .[dev]` reported success, "
            f"but `import vlm_lab` still failed: {exc}"
        ) from exc

    # The clone above is the ONLY source of this checkout, and it can silently
    # be a different revision than the notebook expects. Verify and print the
    # checked-out revision so it is recorded rather than assumed.
    def _git(*args):
        """Run a read-only git command in the checkout, failing loudly."""
        result = subprocess.run(["git", *args], capture_output=True, text=True)
        if result.returncode != 0 or not result.stdout.strip():
            raise RuntimeError(
                f"`git {' '.join(args)}` failed in {os.getcwd()!r} "
                f"(exit code {result.returncode}): {result.stderr.strip()!r}. "
                "The revision check below cannot run, so this cell refuses to "
                "report success rather than printing an unknown revision."
            )
        return result.stdout.strip()

    checked_out_branch = _git("rev-parse", "--abbrev-ref", "HEAD")
    checked_out_head = _git("rev-parse", "--short", "HEAD")
    print(f"Checked out {checked_out_branch} @ {checked_out_head}")

    # This notebook's OWN dependency check (separate from vlm_lab): confirm
    # peft and bitsandbytes -- the two packages the VRAM gate is built on --
    # actually import in this environment, so a missing/broken install shows
    # up here with a clear message instead of as a bare ImportError deep
    # inside `vgc.build_model()` several cells later.
    REQUIRED_DEPENDENCY_MODULES = ("peft", "bitsandbytes")
    _missing_dependencies = [
        name for name in REQUIRED_DEPENDENCY_MODULES
        if importlib.util.find_spec(name) is None
    ]
    if _missing_dependencies:
        raise RuntimeError(
            f"Required dependencies {_missing_dependencies} are not importable, "
            "even though `pip install -e .[dev]` reported success above -- check "
            "pyproject.toml's pins and the pip install log printed earlier."
        )

    # configs/derived_budget.yaml is written by the COMPANION notebook
    # notebooks/01a_closure_measurements.ipynb, not by this one. This is only
    # an early, informational heads-up -- the actual load, with a full and
    # actionable error, happens in the dedicated loader cell below via
    # `vgc.load_derived_budget()`, which is the single source of truth for
    # that error message.
    if not os.path.isfile("configs/derived_budget.yaml"):
        print(
            "NOTE: configs/derived_budget.yaml does not exist yet in this checkout. "
            "The dedicated loader cell below will raise a clear, actionable error "
            "explaining what to run first -- this is just an early heads-up, not "
            "a failure by itself."
        )

    print("Repository cloned, package installed, and required dependencies import successfully.")
else:
    print("Not running on Colab -- skipping repo clone / install (package already installed locally).")


## Hard GPU requirement

4-bit QLoRA and every measurement below require an actual CUDA device --
this notebook is meaningless without one. Unlike
`notebooks/00_environment.ipynb` (which tolerates a CPU-only local run for
structural validation), this notebook asserts on GPU availability
immediately.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False. This notebook measures real GPU "
        "VRAM usage -- the entire point of the docs/proposals/phase1_closure_prereg.md "
        "§8.2 go/no-go gate -- and produces no meaningful result without a GPU. "
        "On Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), "
        "then Runtime -> Restart session, then re-run this notebook top to bottom."
    )

print("CUDA available: True")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
print(f"torch.version.cuda: {torch.version.cuda}")


## Imports and shared gate infrastructure

`scripts/_vram_gate_common.py` (imported as `vgc`) provides:

- `load_derived_budget(path)` -- the `configs/derived_budget.yaml` contract
  loader (this notebook's first substantive cell, below).
- `measured_arm(name)` -- the §8.2 measurement protocol (peak-stat reset, a
  10 Hz background free-memory sampler, and the fixed 1.0 GiB margin GO
  criterion on both the allocator inequality and the whole-device
  min-free-observed inequality).
- `build_base_model(budget)` / `inject_fresh_adapter(...)` / `build_model(budget)`
  -- the frozen §7.3 load → k-bit-prepare → adapter-inject → `use_cache=False`
  order, plus the ADR-012 approval-gate assertions run against the real
  constructed model.
- `build_synthetic_training_batch(...)` / `build_synthetic_eval_prefix(...)`
  -- production-shape dummy tensors (random content, real shapes), including
  `mm_token_type_ids` -- see the note in the next markdown cell.
- `free_cuda_memory()` -- the between-arms `gc.collect()` / `empty_cache()` /
  `reset_peak_memory_stats()` helper. Each arm cell `del`s its own
  model/optimizer/tensor variables itself immediately before calling this --
  `del` inside a helper function cannot remove a variable from the caller's
  scope, only its own, so that step cannot be delegated.


In [ ]:
import gc
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import bitsandbytes as bnb
from torch.amp import GradScaler
from transformers import GenerationConfig

sys.path.insert(0, str(Path("scripts").resolve()))
import _vram_gate_common as vgc  # noqa: E402

SEED = 42
print("Imports OK.")


## Load `configs/derived_budget.yaml` (the companion notebook's contract)

This is the notebook's **first substantive cell**. It loads
`configs/derived_budget.yaml` via `vgc.load_derived_budget()`, which calls
`yaml.safe_load` internally and raises a clear, actionable `FileNotFoundError`
naming `notebooks/01a_closure_measurements.ipynb` if the file is missing --
this notebook does **not** fall back to hardcoded defaults, because doing so
would size arms B and D against invented numbers instead of the measured
production shape, defeating the gate's entire purpose. It also validates the
file's top-level and nested keys match the frozen schema, and cross-checks
`model_revision` against the pre-registered SHA
(`docs/proposals/phase1_closure_prereg.md` §1 / ADR-015).

**A real gap found by inspecting the installed `transformers==5.15.0` source,
not assumed from the spec text:** `Qwen3VLModel.get_rope_index()` /
`compute_3d_position_ids()` **require** an `mm_token_type_ids` tensor
(0=text, 1=image, 2=video) whenever `image_grid_thw` is passed alongside
`input_ids` -- omitting it raises
`ValueError: ... "mm_token_type_ids" is missing ...` at the first forward
call. This field is not listed in this gate's delegated task spec's synthetic
batch field list; `vgc.build_synthetic_training_batch` /
`build_synthetic_eval_prefix` include it. Confirmed by an executed structural
test (a tiny random-init `Qwen3VLForConditionalGeneration`, real config
constants, forward+backward with labels, and `generate()` with
`cache_implementation="static"`, all succeeding only once
`mm_token_type_ids` was added -- and the documented `ValueError` reproducing
exactly when it is omitted).


In [ ]:
derived_budget = vgc.load_derived_budget("configs/derived_budget.yaml")

print("Loaded configs/derived_budget.yaml:")
print(f"  schema_version: {derived_budget['schema_version']}")
print(f"  generated_at_utc: {derived_budget['generated_at_utc']}")
print(f"  model_revision: {derived_budget['model_revision']}")
print(f"  dataset_revision: {derived_budget['dataset_revision']}")
print(f"  processor_size: {derived_budget['processor_size']}")
print(f"  patch_size: {derived_budget['patch_size']}, spatial_merge_size: {derived_budget['spatial_merge_size']}")
print(f"  image_token_ceiling: {derived_budget['image_token_ceiling']}")
print(f"  fixed_prompt_and_template_tokens: {derived_budget['fixed_prompt_and_template_tokens']}")
print(f"  eval_prefix_upper_bound: {derived_budget['eval_prefix_upper_bound']}")
print(f"  max_new_tokens: {derived_budget['max_new_tokens']}")
print(f"  max_seq_len: {derived_budget['max_seq_len']}")
print(f"  lora: {derived_budget['lora']}")


## Arm A — Load

Model load + NF4 quantization + k-bit preparation + fresh LoRA adapter
injection (`vgc.build_model`, §7.3's frozen order). Records the realized
per-module dtype summary, the enabled SDPA backends, GPU identity, and runs
the ADR-012 approval-gate assertions against the real constructed model
(33,030,144 trainable parameters, all under the language tower).

Per §8.2's table, arm A's peak reflects **load only** -- this is the
narrowest of the three model-building arms, included as its own reference
point rather than as a component of arm B/D's (which each build their own
fresh model, reflecting what a genuine fresh "train" or "eval" process would
show as its peak VRAM).


In [ ]:
with vgc.measured_arm("A-load") as arm_a_result:
    model_a, build_info_a = vgc.build_model(derived_budget, adapter_seed=SEED)

print()
print("[Arm A] realized dtype counts:", build_info_a["dtype_counts"])
print("[Arm A] enabled SDPA backends:", build_info_a["sdpa_backend"])
print("[Arm A] GPU:", build_info_a["gpu_name"], "capability:", build_info_a["gpu_capability"])
print("[Arm A] ADR-012 trainable params:", build_info_a["trainable_params"],
      "(expected", derived_budget["lora"]["expected_trainable_params"], ")")
print("[Arm A] ADR-012 real trainable-parameter prefix:", build_info_a["trainable_param_prefix"])

del model_a
vgc.free_cuda_memory()


## Arm B — Train

Fresh model construction, then a full 8-microbatch gradient-accumulation
window per optimizer step (`per_device_train_batch_size=1`,
`gradient_accumulation_steps=8`, §7.3), for **10 optimizer steps total**: 2
discarded warm-up steps, then steps 3..10 timed for the §8.2 step-time
statistic (median). `model.config.use_cache = False` and gradient
checkpointing are already set by `vgc.build_model`.

Optimizer: `bitsandbytes.optim.PagedAdamW8bit` (verified against the pinned
`bitsandbytes==0.50.0`'s actual signature: `params, lr, betas, eps,
weight_decay, amsgrad, optim_bits, args, min_8bit_size`). fp16 autocast +
`torch.amp.GradScaler("cuda")` per §3.4/§7.3 (the pinned `torch==2.13.0`
deprecates the old `torch.cuda.amp.GradScaler(...)` spelling in favor of
`torch.amp.GradScaler("cuda", ...)`, used here).

The synthetic batch is built **once** and reused across all microbatches and
steps (content is irrelevant to a memory/timing gate); this also keeps the
step-time statistic measuring GPU compute/allocator behavior rather than
host-side random-tensor construction overhead.

**Step-time rule (§8.2 step 6):** after the full-budget run, the SAME
procedure runs again at `image_token_ceiling // 2` as the timing baseline,
reusing the same model/optimizer (not a fresh reload) since only wall-clock
timing is being compared, not memory -- a design choice made here to avoid a
second, costly full model reload+requantization on Colab for a
timing-only measurement; stated explicitly as a judgment call rather than
silently assumed. `soft_no_go = statistic > 2.0 * baseline OR statistic > 60s`.
A soft NO-GO does not fail arm B's memory GO/NO-GO, but is reported and
triggers the same §4 fallback-ladder language as a hard NO-GO.


In [ ]:
N_MICROBATCHES = 8  # gradient_accumulation_steps, §7.3
N_OPTIMIZER_STEPS = 10  # 2 warm-up (discarded) + steps 3..10 timed
WARMUP_STEPS = 2

with vgc.measured_arm("B-train") as arm_b_result:
    model_b, build_info_b = vgc.build_model(derived_budget, adapter_seed=SEED)
    model_b.train()

    optimizer_b = bnb.optim.PagedAdamW8bit(
        model_b.parameters(), lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
    )
    scaler_b = GradScaler("cuda")

    torch.manual_seed(SEED)
    batch_full = {
        k: v.to("cuda")
        for k, v in vgc.build_synthetic_training_batch(build_info_b, derived_budget).items()
    }

    step_times_full = []
    for step in range(N_OPTIMIZER_STEPS):
        step_start = vgc.now_s()
        optimizer_b.zero_grad()
        for _microbatch in range(N_MICROBATCHES):
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model_b(**batch_full)
                loss = outputs.loss
            assert torch.isfinite(loss), (
                f"[Arm B] non-finite fp16 loss at step {step}, microbatch {_microbatch}: {loss.item()}"
            )
            scaler_b.scale(loss / N_MICROBATCHES).backward()
        scaler_b.step(optimizer_b)
        scaler_b.update()
        torch.cuda.synchronize()
        step_times_full.append(vgc.now_s() - step_start)

    scale_after = scaler_b.get_scale()

full_budget_stat = vgc.median_step_time(step_times_full, warmup_steps=WARMUP_STEPS)
SCALE_COLLAPSE_THRESHOLD = 1.0  # near-zero => persistent fp16 overflow
scaler_healthy = scale_after > SCALE_COLLAPSE_THRESHOLD

print()
print(f"[Arm B] all {N_OPTIMIZER_STEPS} optimizer steps completed with finite fp16 loss.")
print(f"[Arm B] GradScaler scale after training: {scale_after} "
      f"({'healthy' if scaler_healthy else 'COLLAPSED -- persistent overflow'})")
print(f"[Arm B] median optimizer-step time at full image-token budget "
      f"({derived_budget['image_token_ceiling']} tokens), steps {WARMUP_STEPS + 1}..{N_OPTIMIZER_STEPS}: "
      f"{full_budget_stat:.3f}s")

# --- Half-budget timing baseline: same model/optimizer, timing only (see markdown above). ---
half_budget = derived_budget["image_token_ceiling"] // 2
torch.manual_seed(SEED + 1)
batch_half = {
    k: v.to("cuda")
    for k, v in vgc.build_synthetic_training_batch(
        build_info_b, derived_budget, image_token_budget=half_budget
    ).items()
}

step_times_half = []
for step in range(N_OPTIMIZER_STEPS):
    step_start = vgc.now_s()
    optimizer_b.zero_grad()
    for _microbatch in range(N_MICROBATCHES):
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model_b(**batch_half)
            loss = outputs.loss
        assert torch.isfinite(loss), f"[Arm B baseline] non-finite loss at step {step}: {loss.item()}"
        scaler_b.scale(loss / N_MICROBATCHES).backward()
    scaler_b.step(optimizer_b)
    scaler_b.update()
    torch.cuda.synchronize()
    step_times_half.append(vgc.now_s() - step_start)

half_budget_stat = vgc.median_step_time(step_times_half, warmup_steps=WARMUP_STEPS)
print(f"[Arm B] median optimizer-step time at half image-token budget "
      f"({half_budget} tokens): {half_budget_stat:.3f}s")

soft_no_go = (full_budget_stat > 2.0 * half_budget_stat) or (full_budget_stat > 60.0)
print(f"[Arm B] soft NO-GO (paging/slowness) verdict: {'NO-GO' if soft_no_go else 'OK'} "
      f"(full={full_budget_stat:.3f}s vs. 2x half={2.0 * half_budget_stat:.3f}s, 60s ceiling)")
if soft_no_go:
    print("[Arm B] Soft NO-GO triggers the same §4 fallback-ladder language as a hard NO-GO "
          "(see the final verdict cell).")

del model_b, optimizer_b, scaler_b, batch_full, batch_half
vgc.free_cuda_memory()


## Arm C — Resume (two genuine subprocesses)

Per §8.2, arm C is deliberately implemented as **two separate OS processes**
-- the one exception to "fresh process per arm" being the norm everywhere
else, because a resume that never left the process proves nothing about
reload:

- **`scripts/_vram_gate_c1_save.py`**: builds the model, runs 2 optimizer
  steps on a synthetic batch (same shape as arm B), then saves a full
  trainer-equivalent checkpoint (adapter state dict, optimizer state dict,
  `GradScaler` state, `torch.get_rng_state()` / `torch.cuda.get_rng_state()`,
  and a dummy dataloader-position integer) to a temporary checkpoint
  directory, reporting its own peak memory via `vgc.measured_arm`.
- **`scripts/_vram_gate_c2_resume.py`**: starts cold (a genuinely new
  `sys.executable` process), loads that checkpoint in full, runs **one**
  further optimizer step, and reports its own peak memory.

Both scripts print exactly one JSON line to stdout with their measurement;
this cell parses that line rather than scraping human-readable log text.

**Combining C1 and C2 into one arm-C verdict:** each process's own
`allocator_margin_pass` / `global_free_margin_pass` booleans are computed
against *that process's own* `baseline_free` (correctly paired within one
process). This cell reports **both** the descriptive combined statistics
`§8.2 asks for ("the gate takes the maximum of C1's and C2's peaks") and a
combined GO/NO-GO computed as `C1.go AND C2.go` -- deliberately **not**
re-deriving a single pass/fail from the two processes' differing
`baseline_free` values, since mixing two different baselines into one
inequality would not be sound. This is a judgment call on an
underspecified part of §8.2's combination rule, stated explicitly here
rather than resolved silently.


In [ ]:
import tempfile

ARM_C_CHECKPOINT_DIR = tempfile.mkdtemp(prefix="vram_gate_arm_c_")
print(f"[Arm C] checkpoint dir (outside the repo, never committed): {ARM_C_CHECKPOINT_DIR}")


def _run_arm_c_process(script_name: str) -> dict:
    result = subprocess.run(
        [sys.executable, f"scripts/{script_name}", "--checkpoint-dir", ARM_C_CHECKPOINT_DIR],
        capture_output=True,
        text=True,
    )
    print(f"--- {script_name} stdout ---")
    print(result.stdout)
    if result.stderr:
        print(f"--- {script_name} stderr ---")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{script_name} exited with code {result.returncode}; see stderr above.")
    json_lines = [line for line in result.stdout.strip().splitlines() if line.startswith("{")]
    if not json_lines:
        raise RuntimeError(f"{script_name} did not print a JSON measurement line to stdout.")
    return json.loads(json_lines[-1])


c1_data = _run_arm_c_process("_vram_gate_c1_save.py")
c2_data = _run_arm_c_process("_vram_gate_c2_resume.py")

arm_c_go = bool(c1_data["go"]) and bool(c2_data["go"])
combined_max_reserved_gib = max(c1_data["max_memory_reserved_gib"], c2_data["max_memory_reserved_gib"])
combined_min_free_observed_gib = min(c1_data["min_free_observed_gib"], c2_data["min_free_observed_gib"])

print(f"[Arm C] C1-save GO/NO-GO: {'GO' if c1_data['go'] else 'NO-GO'}")
print(f"[Arm C] C2-resume GO/NO-GO: {'GO' if c2_data['go'] else 'NO-GO'}")
print(f"[Arm C] combined max_memory_reserved across both processes: {combined_max_reserved_gib:.3f} GiB")
print(f"[Arm C] combined min_free_observed across both processes: {combined_min_free_observed_gib:.3f} GiB")
print(f"[Arm C] combined GO/NO-GO (C1 AND C2 both GO): {'GO' if arm_c_go else 'NO-GO'}")


## Arm D — Eval

Fresh model construction, then generation at the `eval_prefix_upper_bound`
input length (image tokens = `image_token_ceiling`, text tokens =
`fixed_prompt_and_template_tokens`), decode length **forced** to the full
configured `max_new_tokens` (`min_new_tokens = max_new_tokens`, so EOS is
effectively suppressed -- the worst *allowed* case rather than an
early-stopped one, per §8.2/§4).

Per §8.2, "adapter-applied" here means the SAME model instance built with a
freshly-**initialized** (not trained) adapter of the registered shape, with
adapters toggled via the real verified PEFT API
(`model.base_model.disable_adapter_layers()` /
`enable_adapter_layers()` -- `PeftModel` in the pinned `peft==0.20.0` has no
`enable_adapters()`/`disable_adapters()` method; these are what its own
`disable_adapter()` context manager calls internally for a LoRA model,
confirmed by reading the installed source).

Both conditions run inside **one** `measured_arm("D-eval")` block with no
intermediate peak-stat reset, so `torch.cuda.max_memory_allocated()` /
`max_memory_reserved()` read right after the Base condition give the
**cumulative** peak through Base, and the same read after the
adapter-applied condition gives the **combined** peak through both (peak
stats are monotonically non-decreasing without a reset) -- satisfying both
"report peak across both" and "report each condition's numbers separately"
without needing to manually recombine two independently-reset
measurements.

`cache_implementation="static"` (§5.4) is attempted first; if
`generate()` raises for it, the exact error is reported and the run falls
back to the default cache implementation, flagged explicitly as a deviation
from §8.2's spec rather than silently substituted.


In [ ]:
# Confirmed against the real pinned tokenizer (Qwen/Qwen3-VL-4B-Instruct @
# the ADR-015 revision), not guessed. Since min_new_tokens == max_new_tokens
# below, generation never stops early regardless of this value -- these ids
# only need to be valid, not semantically meaningful, for this memory gate.
EOS_TOKEN_ID = 151645  # <|im_end|>
PAD_TOKEN_ID = 151643  # <|endoftext|>

with vgc.measured_arm("D-eval") as arm_d_result:
    model_d, build_info_d = vgc.build_model(derived_budget, adapter_seed=SEED)
    model_d.eval()

    torch.manual_seed(SEED)
    eval_batch = {
        k: v.to("cuda")
        for k, v in vgc.build_synthetic_eval_prefix(build_info_d, derived_budget).items()
    }

    max_new_tokens = derived_budget["max_new_tokens"]
    gen_config = GenerationConfig(
        do_sample=False,
        num_beams=1,
        min_new_tokens=max_new_tokens,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        cache_implementation="static",
        eos_token_id=EOS_TOKEN_ID,
        pad_token_id=PAD_TOKEN_ID,
    )

    cache_implementation_used = "static"
    model_d.base_model.disable_adapter_layers()
    try:
        with torch.no_grad():
            base_generated = model_d.generate(**eval_batch, generation_config=gen_config)
    except Exception as exc:
        print(f"[Arm D] cache_implementation='static' FAILED: {type(exc).__name__}: {exc}")
        print("[Arm D] Falling back to the default cache implementation -- "
              "THIS DEVIATES FROM §8.2's spec and is reported, not silently absorbed.")
        cache_implementation_used = f"dynamic (fallback after {type(exc).__name__})"
        gen_config.cache_implementation = None
        with torch.no_grad():
            base_generated = model_d.generate(**eval_batch, generation_config=gen_config)

    peak_allocated_after_base = torch.cuda.max_memory_allocated()
    peak_reserved_after_base = torch.cuda.max_memory_reserved()
    print(f"[Arm D] Base condition (adapter disabled) -- cumulative peak through this point: "
          f"allocated={peak_allocated_after_base / vgc.GIB:.3f} GiB, "
          f"reserved={peak_reserved_after_base / vgc.GIB:.3f} GiB")

    model_d.base_model.enable_adapter_layers()
    with torch.no_grad():
        adapter_generated = model_d.generate(**eval_batch, generation_config=gen_config)

    peak_allocated_after_adapter = torch.cuda.max_memory_allocated()
    peak_reserved_after_adapter = torch.cuda.max_memory_reserved()
    print(f"[Arm D] Adapter-applied condition -- cumulative peak through this point "
          f"(= this arm's combined peak): "
          f"allocated={peak_allocated_after_adapter / vgc.GIB:.3f} GiB, "
          f"reserved={peak_reserved_after_adapter / vgc.GIB:.3f} GiB")

print()
print(f"[Arm D] cache_implementation actually used: {cache_implementation_used}")
print(f"[Arm D] Base output length: {base_generated.shape[-1]} "
      f"(input {eval_batch['input_ids'].shape[-1]} + up to {max_new_tokens} new)")
print(f"[Arm D] Adapter-applied output length: {adapter_generated.shape[-1]}")

del model_d, eval_batch, base_generated, adapter_generated
vgc.free_cuda_memory()


## Final verdict

Aggregates all four arms' hard memory GO/NO-GO plus arm B's soft-NO-GO
timing verdict. `GATE VERDICT` is `NO-GO` if **any** arm's hard memory
criterion fails; a soft NO-GO alone does not flip the headline verdict but
still triggers the same §4 fallback-ladder guidance, printed either way it
fires. The full result -- all four arms' numbers, both GO/NO-GO booleans,
the ADR-012 trainable-parameter evidence, and the realized dtype/SDPA/GPU
info -- is written to `results/vram_gate_verdict.json`.


In [ ]:
arm_results_summary = {
    "A-load": arm_a_result["measurement"].to_dict(),
    "B-train": {
        **arm_b_result["measurement"].to_dict(),
        "step_time_full_budget_s": full_budget_stat,
        "step_time_half_budget_s": half_budget_stat,
        "soft_no_go": soft_no_go,
        "grad_scaler_final_scale": scale_after,
        "grad_scaler_healthy": scaler_healthy,
    },
    "C-resume": {
        "c1_save": c1_data,
        "c2_resume": c2_data,
        "combined_max_memory_reserved_gib": combined_max_reserved_gib,
        "combined_min_free_observed_gib": combined_min_free_observed_gib,
        "go": arm_c_go,
    },
    "D-eval": {
        **arm_d_result["measurement"].to_dict(),
        "cache_implementation_used": cache_implementation_used,
        "peak_allocated_after_base_gib": peak_allocated_after_base / vgc.GIB,
        "peak_reserved_after_base_gib": peak_reserved_after_base / vgc.GIB,
        "peak_allocated_after_adapter_gib": peak_allocated_after_adapter / vgc.GIB,
        "peak_reserved_after_adapter_gib": peak_reserved_after_adapter / vgc.GIB,
    },
}

hard_go = all(
    [
        arm_a_result["measurement"].go,
        arm_b_result["measurement"].go,
        arm_c_go,
        arm_d_result["measurement"].go,
    ]
)
overall_verdict = "GO" if hard_go else "NO-GO"

print("=" * 70)
print(f"GATE VERDICT: {overall_verdict}")
print("=" * 70)
print(f"  A-load   GO/NO-GO: {'GO' if arm_results_summary['A-load']['go'] else 'NO-GO'}")
print(f"  B-train  GO/NO-GO: {'GO' if arm_results_summary['B-train']['go'] else 'NO-GO'} "
      f"(soft NO-GO: {soft_no_go})")
print(f"  C-resume GO/NO-GO: {'GO' if arm_results_summary['C-resume']['go'] else 'NO-GO'}")
print(f"  D-eval   GO/NO-GO: {'GO' if arm_results_summary['D-eval']['go'] else 'NO-GO'}")

if (not hard_go) or soft_no_go:
    print()
    print("FALLBACK LADDER (docs/proposals/phase1_closure_prereg.md §4): reduce "
          "processor_size.longest_edge to 524_288, then 262_144 if still failing; "
          "re-measure this gate and re-record before any performance output is observed.")

verdict_artifact = {
    "schema_version": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_revision": derived_budget["model_revision"],
    "dataset_revision": derived_budget["dataset_revision"],
    "gate_verdict": overall_verdict,
    "arms": arm_results_summary,
    "adr_012_evidence": {
        "trainable_params": build_info_a["trainable_params"],
        "expected_trainable_params": derived_budget["lora"]["expected_trainable_params"],
        "trainable_param_prefix": build_info_a["trainable_param_prefix"],
    },
    "realized_environment": {
        "dtype_counts": build_info_a["dtype_counts"],
        "sdpa_backend": build_info_a["sdpa_backend"],
        "gpu_name": build_info_a["gpu_name"],
        "gpu_capability": build_info_a["gpu_capability"],
    },
    "arm_isolation_deviation": (
        "Arms A, B, D ran within one shared notebook kernel (with explicit "
        "del + gc.collect() + empty_cache() + reset_peak_memory_stats() between "
        "them) rather than in fully separate OS processes, per §8.2 step 1's "
        "frozen protocol -- an approximation, not full isolation. Arm C is the "
        "one exception: it genuinely used two separate subprocesses "
        "(scripts/_vram_gate_c1_save.py, scripts/_vram_gate_c2_resume.py)."
    ),
}

Path("results").mkdir(parents=True, exist_ok=True)
verdict_path = Path("results/vram_gate_verdict.json")
with verdict_path.open("w", encoding="utf-8") as f:
    json.dump(verdict_artifact, f, indent=2, default=str)

print()
print(f"Full verdict written to {verdict_path.resolve()}")
